# Day 1 실습 — ChatPromptTemplate·LCEL 체인과 프롬프트 설계

**목표**: 프롬프트 양식을 만들어 체인으로 연결하고, 역할·지시문·맥락·예시를 하나씩 더하며 답 품질 변화를 직접 확인한다.
**구성**: Part 1 첫 체인 완성·오류 다루기 → Part 2 설계 요소 실험(+다른 도메인) → Part 3 미니 프로젝트(+나만의 캐릭터 챗봇)

> **참고:** 실습 전 가상환경 활성화, `.env`의 OpenAI API 키, 패키지 설치(`uv sync`)를 확인한다.

## 0. 환경 준비·모델 생성

필요한 도구를 불러오고 모델·파서 객체를 만든다. 이 셀은 노트북 전체에서 한 번만 실행한다.

<details>
<summary>각 import의 역할</summary>

- `ChatOpenAI`: OpenAI 채팅 모델을 부르는 객체다.
- `ChatPromptTemplate`: 프롬프트 양식을 만드는 도구다.
- `StrOutputParser`: 답을 순수 문자열로 정리하는 파서다.
- `load_dotenv`: `.env`의 API 키를 불러온다.
</details>

In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()
print("준비 완료:", llm.model_name)

## Part 1. 첫 체인 완성

완성 코드를 직접 쳐서 `prompt | llm | parser` 체인을 처음부터 만든다.

이 체인은 사실 **두 도메인을 오가는 번역 루프**다. 사람이 원하는 것(**사용자 도메인**)과 모델이 다음 글자를 예측하며 이어 쓰는 문서(**모델 도메인**)는 서로 다른 언어이고, 애플리케이션 개발자의 일은 이 둘 사이를 번역하는 것이다.

- `prompt`: 사용자 도메인 → 모델 도메인 (**순방향 번역**) — 질문을 "모델이 이어 쓰고 싶어지는 문서" 형태로 바꾼다
- `llm`: 모델 도메인 안에서의 완성 (다음 글자를 예측해 이어 쓴다)
- `parser`: 모델 도메인 → 사용자 도메인 (**역방향 번역**) — 모델의 텍스트 출력을 프로그램이 쓸 수 있는 형태로 되돌린다

아래 1-1~1-4에서 이 세 조각을 하나씩 만들어본다. Day03(Structured Output)은 이 **역방향 번역을 더 정교하게 만드는 장**이라고 볼 수 있다.

### 1-1. 모델 직접 호출

양식 없이 모델에 질문을 바로 보내 본다. 답은 **메시지 객체**로 온다 (아래 `type`으로 확인).

In [ ]:
# TODO: llm.invoke(...)로 질문을 보내고 답을 받으세요
answer = None

print(answer.content)
print("타입:", type(answer))   # 메시지 객체

### 1-2. ChatPromptTemplate·변수 바인딩

빈칸 `{topic}`이 있는 양식을 만들고, 값을 채워 어떤 메시지가 만들어지는지 확인한다.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 비전공자에게 친절히 설명하는 강사다."),

    # TODO: human 메시지에 빈칸 {topic}을 넣어 질문을 완성하세요
    None,
])

# TODO: 빈칸 {topic}에 "API"를 채워 실행하세요
messages = None

print(messages)

### 1-3. StrOutputParser로 답 정리

파서를 쓰기 전과 후의 **결과 타입 차이**를 눈으로 비교한다.

> **참고:** 파서 전은 메시지 객체, 파서 후는 바로 출력·저장할 수 있는 텍스트다 (langchain 1.x에서는 타입이 `TextAccessor`로 나오지만 문자열처럼 쓸 수 있다).

In [ ]:
raw = llm.invoke(prompt.invoke({"topic": "API"}))
print("파서 전 타입:", type(raw))

# TODO: parser로 raw를 정리하세요
clean = None

print("파서 후 타입:", type(clean))
print(clean)

### 1-4. 체인 완성

`prompt | llm | parser`를 파이프로 연결해 한 줄로 실행한다.

In [ ]:
# TODO: prompt, llm, parser를 파이프(|)로 연결하세요
chain = None

print(chain.invoke({"topic": "API"}))

### 1-5. 재사용 확인

양식은 그대로 두고 값만 바꿔 반복 실행한다. 양식 재사용의 이점을 체감한다.

In [ ]:
# TODO: topic 3개를 넣어 반복 실행하세요
for topic in ["____", "____", "____"]:
    print(f"[{topic}]", chain.invoke({"topic": topic}))

### 1-6. Runnable 인터페이스 — batch로 한 번에 처리하기

`prompt`·`llm`·`parser`·`chain`은 모두 같은 Runnable 규칙을 따른다. `invoke`를 여러 번 부르는 대신 `batch`로 입력을 한 번에 묶어 보낼 수 있다. 결과는 1-5의 반복문과 같다.

In [ ]:
# TODO: chain.batch(...)에 topic 3개를 리스트로 넣어 한 번에 실행하세요
results = chain.batch(None)
for r in results:
    print(r)

### 1-7. 오류 다뤄보기 — 모델명을 잘못 쓰면?

일부러 오류를 내고 메시지를 읽는 연습이다. 전부 `try/except`로 감싸 오류 종류만 확인한다.

In [ ]:
try:
    # TODO: model 이름을 일부러 틀리게 써서 오류를 내보세요 (예: gpt-4o-mno)
    bad_llm = None
    
    print(bad_llm.invoke("안녕").content)
except Exception as e:
    print("오류 종류:", type(e).__name__)
    print(str(e)[:200])

### 1-8. 오류 다뤄보기 — parser를 빼면?

메시지 객체에는 `.upper()` 같은 문자열 메서드가 없다.

In [ ]:
try:
    # TODO: parser를 빼고 체인을 연결하세요
    no_parser_chain = None
    
    result = no_parser_chain.invoke({"topic": "API"})
    print(result.upper())  # 메시지 객체에는 upper()가 없다
except Exception as e:
    print("오류 종류:", type(e).__name__)
    print(str(e)[:200])

### 1-9. 오류 다뤄보기 — API 키가 틀리면?

실제 키(환경변수)는 전혀 건드리지 않는다 — ChatOpenAI(api_key=...)에 일부러 틀린 키를 직접 넣어 인증 오류만 확인한다(복구 단계 자체가 필요 없다).

In [ ]:
try:
    # TODO: api_key="____"처럼 일부러 틀린 키를 넣어 ChatOpenAI를 만드세요 (예: sk-invalid)
    bad_key_llm = ChatOpenAI(model="gpt-4o-mini", api_key="____")
    print(bad_key_llm.invoke("안녕").content)
except Exception as e:
    print("오류 종류:", type(e).__name__)
    print(str(e)[:200])

## Part 2. 설계 요소 실험

같은 질문 하나에 역할·지시문·맥락·예시를 하나씩 더하며 답 변화를 관찰한다. 공통 질문: **"환불 정책이 궁금해요."**

### 2-1. 기준선(V1) — 질문만

아무 요소 없이 질문만 보낸다. 회사 정보가 없어 일반론으로 답한다.

In [ ]:
question = "환불 정책이 궁금해요."

# TODO: 질문만 담은 기본 프롬프트를 만드세요
v1 = None

print((v1 | llm | parser).invoke({"question": question}))

### 2-2. 역할(Role) 부여

system 메시지로 AI가 누구인지 정한다. 답의 톤이 바뀐다.

In [ ]:
v2 = ChatPromptTemplate.from_messages([
    # TODO: system 메시지로 역할을 정하세요 (예: 친절한 고객센터 상담사)
    None,
    ("human", "{question}"),
])
print((v2 | llm | parser).invoke({"question": question}))

### 2-3. 지시문(Instruction) 명확화

어떻게 답할지 지시한다. 형식·길이가 통제된다.

In [ ]:
v3 = ChatPromptTemplate.from_messages([
    # TODO: 역할 뒤에 형식 지시를 추가하세요 (예: 3문장 이내, 존댓말)
    None,
    
    ("human", "{question}"),
])
print((v3 | llm | parser).invoke({"question": question}))

### 2-4. 맥락(Context) 주입

참고 자료를 함께 넣는다. 답이 근거 기반으로 정확해진다.

In [ ]:
context = """
[환불 규정]
- 구매 후 7일 이내 미개봉 상품만 환불 가능
- 환불은 영업일 기준 3일 내 처리
- 디지털 상품은 환불 불가
"""

v4 = ChatPromptTemplate.from_messages([
    # TODO: system 끝에 "규정을 근거로 답" 지시와 {context}를 넣으세요
    None,
    ("human", "{question}"),
])
print((v4 | llm | parser).invoke({"context": context, "question": question}))

### 2-5. few-shot 예시 제공

입력·출력 예시를 보여 준다. 형식 일관성이 높아진다.

> **참고:** 지금까지 Part 2에서 채워온 역할·지시문·맥락(`context`)·예시는 전부 **정적 콘텐츠**다 — 어떤 질문이 오든 프롬프트에 고정으로 박혀 있다. 실무에서는 요청마다 달라지는 **동적 콘텐츠**(예: 그때그때 검색해서 가져오는 문서)도 필요하다 — 방금 2-4의 `context`를 하드코딩된 문자열 대신 실시간 검색 결과로 바꾸면 그게 바로 M3(Day12)에서 배울 RAG다.

In [ ]:
fs = ChatPromptTemplate.from_messages([
    ("system", "너는 친절한 고객센터 상담사다. 아래 예시 형식으로 답한다."),
    ("human", "교환 되나요?"),
    
    # TODO: 위 질문에 대한 모범 답변 예시를 ai 메시지로 작성하세요
    None,
    
    ("human", "{question}"),
])
print((fs | llm | parser).invoke({"question": question}))

### 2-6. 조합 — 역할+지시문+맥락+예시

네 요소를 모두 넣어 최상의 답을 만든다.

In [ ]:
combo = ChatPromptTemplate.from_messages([
    # TODO: V4(역할+지시문+맥락)에 few-shot 예시까지 합쳐 완성하세요
    None,
    ("human", "교환 되나요?"),
    
    None,
    ("human", "{question}"),
])
print((combo | llm | parser).invoke({"context": context, "question": question}))

### 요소별 효과 정리

| 요소 | 주로 좋아지는 것 |
| --- | --- |
| 역할 | 톤·태도 |
| 지시문 | 형식·길이 |
| 맥락 | 정확성·근거 |
| 예시 | 형식 일관성 |

### 2-6-보충. 질문이 모호하면? — 명확화 되묻기

지금까지 다룬 질문은 항상 명확했다. 하지만 실제로는 정보가 부족해 규정만으로는 답할 수 없는 질문도 들어온다. 이럴 땐 곧바로(어쩌면 틀린) 답을 내놓는 대신, 모델이 스스로 무엇을 더 알아야 하는지 되묻게 만들 수 있다.

In [ ]:
clarify_prompt = ChatPromptTemplate.from_messages([
    # TODO: 아래 규정({context})을 참고하되, 질문이 모호해 규정만으로 답할 수 없으면 무엇을 더 알아야 하는지
    #       되묻고, 규정으로 답할 수 있으면 바로 답하도록 지시하는 system 메시지를 작성하세요
    None,
    ("human", "{question}"),
])

print("--- 모호한 질문(상품이 특정 안 됨) ---")
print((clarify_prompt | llm | parser).invoke({"context": context, "question": "이 상품 환불 되나요?"}))
print()
print("--- 명확한 질문(규정 자체를 물음) ---")
print((clarify_prompt | llm | parser).invoke({"context": context, "question": "환불 규정이 어떻게 되나요?"}))

## Part 2-확장. 다른 도메인에 적용하기 — 모의면접 코치 AI

같은 네 가지 요소(역할·지시문·맥락·예시)가 **도메인이 완전히 달라져도** 똑같이 통하는지 확인한다. 이번 주제는 취업 준비생을 위한 모의면접 코치다. 공통 질문: **"백엔드 개발자 면접을 준비하고 있어요. 예상 질문을 알려주세요."**

### 2-7. 기준선(V1) — 질문만

In [ ]:
question2 = "백엔드 개발자 면접을 준비하고 있어요. 예상 질문을 알려주세요."

v1_coach = ChatPromptTemplate.from_messages([("human", "{question}")])
print((v1_coach | llm | parser).invoke({"question": question2}))

### 2-8. 역할+지시문 — 면접 코치 캐릭터 잡기

이번에는 예시 없이 직접 채운다. 현직 개발자 출신 모의면접 코치 역할과, 질문을 목록으로 정리하라는 지시를 함께 넣는다.

In [ ]:
v2_coach = ChatPromptTemplate.from_messages([
    # TODO: 현직 개발자 출신 모의면접 코치 역할과, 질문을 목록으로 정리하라는 지시를 함께 넣으세요
    None,
    
    ("human", "{question}"),
])
print((v2_coach | llm | parser).invoke({"question": question2}))

### 2-9. 맥락 추가 — 지원자 이력

실제 지원자의 경력·기술 스택을 맥락으로 넣어, 눈높이에 맞는 질문이 나오게 한다.

In [ ]:
# TODO: 경력·사용 기술을 채우세요
candidate_info = """
[지원자 이력]
- 경력: ____
- 사용 기술: ____
"""

v3_coach = ChatPromptTemplate.from_messages([
    # TODO: system 끝에 "지원자 이력에 맞춰 난이도·주제 조정" 지시와 {candidate_info}를 넣으세요
    None,
    
    ("human", "{question}"),
])
print((v3_coach | llm | parser).invoke({"candidate_info": candidate_info, "question": question2}))

### 2-10. 예시(few-shot) 추가 — 답변 형식 고정

In [ ]:
v4_coach = ChatPromptTemplate.from_messages([
    ("system", "너는 15년차 현직 백엔드 개발자 출신 모의면접 코치다. 아래 예시 형식으로 답한다."),
    ("human", "프론트엔드 개발자 면접 예상 질문 알려주세요."),
    # TODO: 위 질문에 대한 모범 답변 예시를 ai 메시지로 작성하세요 (질문 → 힌트 형식)
    None,
    
    ("human", "{question}"),
])
print((v4_coach | llm | parser).invoke({"question": question2}))

### 관찰 정리

- 상담사 도메인과 모의면접 코치 도메인 모두에서, 어떤 요소가 똑같이 중요했는가?
- 도메인이 바뀌면서 새로 신경 써야 했던 부분은 무엇인가?

## Part 3. 미니 프로젝트 — 프롬프트 4종 비교

하나의 질문을 V1(질문만)~V4(역할+지시문+맥락)로 쌓아 품질 변화를 직접 측정한다.

### 3-1. 질문·조건 정의

In [ ]:
question = "제주도 3일 여행 일정을 추천해줘."
context = """
[여행자 조건]
- 예산: 1인 40만원
- 이동수단: 대중교통만
- 관심사: 자연 경관, 카페
"""

### 3-2. V1~V4 프롬프트 정의

In [ ]:
# V1 질문만 (완성)
v1 = ChatPromptTemplate.from_messages([("human", "{question}")])

# V2 + 역할
v2 = ChatPromptTemplate.from_messages([
    # TODO: 역할을 정하세요 (예: 제주 여행 전문 플래너)
    None,
    ("human", "{question}"),
])

# V3 + 지시문
v3 = ChatPromptTemplate.from_messages([
    # TODO: 역할 + 형식 지시(Day별·하루 3곳·목록)를 넣으세요
    None,
    ("human", "{question}"),
])

# V4 + 맥락
v4 = ChatPromptTemplate.from_messages([
    # TODO: 역할 + 지시문 + {context} 반영을 모두 넣으세요
    None,
    ("human", "{question}"),
])

### 3-3. 반복 실행 — 네 버전 비교 출력

In [ ]:
for name, tmpl in [("V1", v1), ("V2", v2), ("V3", v3), ("V4", v4)]:
    chain = tmpl | llm | parser
    inputs = {"question": question}
    if name == "V4":
        inputs["context"] = context
    print(f"===== {name} =====")
    print(chain.invoke(inputs))
    print()

### 품질 비교표 (직접 채우기)

네 답을 읽고 상/중/하로 평가한다.

| 버전 | 정확성 | 형식 준수 | 톤 적합 | 총평 |
| --- | --- | --- | --- | --- |
| V1 | | | | |
| V2 | | | | |
| V3 | | | | |
| V4 | | | | |

**확인 질문**
- 어떤 요소를 더했을 때 품질이 가장 크게 올랐는가?
- 효과가 작았던 요소는 무엇이며, 왜 그럴까?

## Part 3-확장. 나만의 AI 캐릭터 챗봇

좋아하는 캐릭터(만화·영화·게임 등)를 하나 정해, V1~V4로 그 캐릭터처럼 답하는 챗봇을 직접 만든다. 진행 방식은 Part 3과 같다. 아래는 강사 예시(명탐정 코난)이며, **자신만의 캐릭터로 바꿔서 진행한다.**

### 캐릭터·질문 정의

In [ ]:
# TODO: 좋아하는 캐릭터로 바꿔서 채우세요
character = "____"
question3 = "____"
character_info = """
[캐릭터 설정]
- 말버릇: ____
- 성격: ____
"""

### V1~V4 정의

In [ ]:
# V1 질문만 (완성)
v1_c = ChatPromptTemplate.from_messages([("human", "{question}")])

# V2 + 역할
v2_c = ChatPromptTemplate.from_messages([
    # TODO: character 변수를 이용해 역할을 정하세요
    None,
    ("human", "{question}"),
])

# V3 + 지시문
v3_c = ChatPromptTemplate.from_messages([
    # TODO: 역할 + 캐릭터 말투·길이 지시를 넣으세요
    None,
    ("human", "{question}"),
])

# V4 + 맥락(캐릭터 설정)
v4_c = ChatPromptTemplate.from_messages([
    # TODO: 역할 + 지시문 + {character_info} 반영을 모두 넣으세요
    None,
    ("human", "{question}"),
])

### 실행·비교

In [ ]:
for name, tmpl in [("V1", v1_c), ("V2", v2_c), ("V3", v3_c), ("V4", v4_c)]:
    chain = tmpl | llm | parser
    inputs = {"question": question3}
    if name == "V4":
        inputs["character_info"] = character_info
    print(f"===== {name} =====")
    print(chain.invoke(inputs))
    print()

### 품질 비교표 (직접 채우기)

| 버전 | 캐릭터다움 | 형식 준수 | 총평 |
| --- | --- | --- | --- |
| V1 | | | |
| V2 | | | |
| V3 | | | |
| V4 | | | |

**확인 질문**
- 이번에는 어떤 요소가 가장 캐릭터를 살렸는가?
- Part 3(제주 여행)의 결과와 비교하면 무엇이 같고 무엇이 달랐는가?

## 확인 문제

1. `system`·`human`·`ai` 메시지는 각각 누구의 말이며, 우선순위는 어떻게 되는가?
2. 파서를 뺐을 때(1-1)와 넣었을 때(1-3) 출력 타입은 어떻게 달랐는가?
3. `invoke`를 여러 번 부르는 것과 `batch`로 한 번에 처리하는 것은 결과가 같은가? 무엇이 다른가?
4. 1-7~1-9에서 만든 세 오류는 각각 원인이 무엇이었는가?
5. 같은 질문에 역할만 추가했을 때와 지시문까지 추가했을 때, 달라지는 지점이 각각 다른 이유는?
6. 맥락(Context)을 넣으면 왜 환각(hallucination)이 줄어드는가?
7. Part 2와 Part 2-확장에서, 도메인이 달라져도 변하지 않았던 것은 무엇인가?
8. Part 3과 Part 3-확장에서 어떤 요소를 더했을 때 품질이 가장 크게 올랐는가?